# Experiment 2: Effects of Choice on the Illusion of Control

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats

from dotenv import load_dotenv
load_dotenv()

try:
    from pandas.io.formats.style import Styler
except Exception:
    Styler = object

## Данные

#### Чтение

In [2]:
RESULTS_DIR_PATH = os.environ.get('RESULTS_FOLDER')

In [25]:
RUN_ID = 'familiarity_choice_full'
EXPERIMENT_ID = 'familiarity_choice'

In [26]:
RESULTS_CSV_PATH = '.' + RESULTS_DIR_PATH + RUN_ID + '/' + EXPERIMENT_ID + '/results.csv'

In [27]:
df = pd.read_csv(RESULTS_CSV_PATH)

In [28]:
df.head(10)

,gender,model_id,would_buy_lottery_ticket,chosen_ticket,would_exchange_lottery_ticket,experiment_id,variation_id,condition_id,iteration_id
0,male,gpt_5_nano,False,Diego Maradona,True,familiarity_choice,rephrasing,choice_familiarity,0
1,male,gpt_5_nano,False,Lionel Messi,True,familiarity_choice,rephrasing,choice_familiarity,0
2,male,gpt_5_nano,False,Lionel Messi,True,familiarity_choice,rephrasing,choice_familiarity,0
3,male,gpt_5_nano,False,David Beckham,True,familiarity_choice,rephrasing,choice_familiarity,0
4,male,gpt_5_nano,False,Diego Maradona,True,familiarity_choice,rephrasing,choice_familiarity,0
5,male,gpt_5_nano,False,NaN,True,familiarity_choice,rephrasing,no_choice_familiarity,0
6,male,gpt_5_nano,False,NaN,True,familiarity_choice,rephrasing,no_choice_familiarity,0
7,male,gpt_5_nano,False,NaN,True,familiarity_choice,rephrasing,no_choice_familiarity,0
8,male,gpt_5_nano,False,NaN,True,familiarity_choice,rephrasing,no_choice_familiarity,0
9,male,gpt_5_nano,False,NaN,True,familiarity_choice,rephrasing,no_choice_familiarity,0


In [29]:
df.shape

(1800, 9)

#### Подготовка

In [30]:
df.isna().sum()

gender                             0
model_id                           0
would_buy_lottery_ticket           0
chosen_ticket                    900
would_exchange_lottery_ticket      0
experiment_id                      0
variation_id                       0
condition_id                       0
iteration_id                       0
dtype: int64

- Пустой `chosen_ticket` — это норм, потому что в `no_choice` выбора нет. 3420 — как раз половина от 6840
- `would_buy_lottery_ticket` и `requested_lottery_ticket_price` не должны быть пустыми!

In [20]:
df.groupby(by=['variation_id', 'model_id']).count()

gender  bet_1  bet_2  bet_3  bet_4  \
variation_id     model_id                                                  
default          deepseek_v3.2           108    108    108    108    108   
                 gpt_5                   108    108    108    108    108   
                 gpt_5_mini              108    108    108    108    108   
                 gpt_5_nano              108    108    108    108    108   
                 qwen_3_max_thinking     108    108    108    108    108   
                 qwen_turbo              108    108    108    108    108   
fully_rational   deepseek_v3.2           108    108    108    108    108   
                 gpt_5                   108    108    108    108    108   
                 gpt_5_mini              108    108    108    108    108   
                 gpt_5_nano              108    108    108    108    108   
                 qwen_3_max_thinking     108    108    108    108    108   
                 qwen_turbo              108    108    108    108    108   
higher_emphasis  deepseek_v3.2           108    108    108    108    108   
                 gpt_5                   108    108    108    108    108   
                 gpt_5_mini              108    108    108    108    108   
                 gpt_5_nano              108    108    108    108    108   
                 qwen_3_max_thinking     108    108    108    108    108   
                 qwen_turbo              108    108    108    108    108   
lower_emphasis   deepseek_v3.2           108    108    108    108    108   
                 gpt_5                   108    108    108    108    108   
                 gpt_5_mini              108    108    108    108    108   
                 gpt_5_nano              108    108    108    108    108   
                 qwen_3_max_thinking     108    108    108    108    108   
                 qwen_turbo              108    108    108    108    108   
rephrasing       deepseek_v3.2           108    108    108    108    108   
                 gpt_5                   108    108    108    108    108   
                 gpt_5_mini              108    108    108    108    108   
                 gpt_5_nano              108    108    108    108    108   
                 qwen_3_max_thinking     108    108    108    108    108   
                 qwen_turbo              108    108    108    108    108   
without_emphasis deepseek_v3.2           108    108    108    107    107   
                 gpt_5                   108    108    108    108    108   
                 gpt_5_mini              108    108    108    108    108   
                 gpt_5_nano              108    108    108    108    108   
                 qwen_3_max_thinking     108    108    108    108    108   
                 qwen_turbo              108    108    108    108    108   

                                      competence  experiment_id  condition_id  \
variation_id     model_id                                                       
default          deepseek_v3.2               108            108           108   
                 gpt_5                       108            108           108   
                 gpt_5_mini                  108            108           108   
                 gpt_5_nano                  108            108           108   
                 qwen_3_max_thinking         108            108           108   
                 qwen_turbo                  108            108           108   
fully_rational   deepseek_v3.2               108            108           108   
                 gpt_5                       108            108           108   
                 gpt_5_mini                  108            108           108   
                 gpt_5_nano                  108            108           108   
                 qwen_3_max_thinking         108            108           108   
                 qwen_turbo                  108            108           108   
higher_emphasis

In [24]:
df[(df['variation_id'] == 'without_emphasis') & (df['model_id'] == 'deepseek_v3.2')].groupby(by=['iteration_id'])[['bet_1', 'bet_2', 'bet_3', 'bet_4']].mean()

,bet_1,bet_2,bet_3,bet_4
iteration_id,,,,
0,6.666667,9.500000,10.257143,13.628571
1,8.277778,10.916667,11.805556,14.416667
2,9.805556,9.888889,11.861111,13.000000


Финальная проверка

In [51]:
df[['gender', 'would_buy_lottery_ticket', 'requested_lottery_ticket_price']].isna().sum().sum()

KeyError: "['requested_lottery_ticket_price'] not in index"

In [54]:
df[df['condition_id'] == 'no_choice_familiarity']

,gender,model_id,would_buy_lottery_ticket,chosen_ticket,would_exchange_lottery_ticket,experiment_id,variation_id,condition_id,iteration_id
5,male,gpt_5_nano,False,Thank you.,True,familiarity_choice,rephrasing,no_choice_familiarity,0
6,male,gpt_5_nano,False,I'll take it.,True,familiarity_choice,rephrasing,no_choice_familiarity,0
7,male,gpt_5_nano,False,Mr. Brown: Thanks.,True,familiarity_choice,rephrasing,no_choice_familiarity,0
8,male,gpt_5_nano,False,Thank you.,True,familiarity_choice,rephrasing,no_choice_familiarity,0
9,male,gpt_5_nano,False,Thanks.,True,familiarity_choice,rephrasing,no_choice_familiarity,0
...,...,...,...,...,...,...,...,...,...
1685,male,qwen_3_max_thinking,True,___,False,familiarity_choice,fully_rational,no_choice_familiarity,1
1686,male,qwen_3_max_thinking,True,___,False,familiarity_choice,fully_rational,no_choice_familiarity,1
1687,male,qwen_3_max_thinking,True,___,True,familiarity_choice,fully_rational,no_choice_familiarity,1
1688,male,qwen_3_max_thinking,True,___,True,familiarity_choice,fully_rational,no_choice_familiarity,1


In [56]:
df[(df['condition_id'] == 'no_choice_familiarity') | (df['condition_id'] == 'no_choice_no_familiarity')][['chosen_ticket']].isna().sum().sum()

np.int64(6)

In [52]:
df

,gender,model_id,would_buy_lottery_ticket,chosen_ticket,would_exchange_lottery_ticket,experiment_id,variation_id,condition_id,iteration_id
0,male,gpt_5_nano,False,Lionel Messi,True,familiarity_choice,rephrasing,choice_familiarity,0
1,male,gpt_5_nano,False,Lionel Messi,True,familiarity_choice,rephrasing,choice_familiarity,0
2,male,gpt_5_nano,False,Lionel Messi,True,familiarity_choice,rephrasing,choice_familiarity,0
3,male,gpt_5_nano,False,Lionel Messi,True,familiarity_choice,rephrasing,choice_familiarity,0
4,male,gpt_5_nano,False,Neymar Jr,True,familiarity_choice,rephrasing,choice_familiarity,0
...,...,...,...,...,...,...,...,...,...
1695,male,qwen_3_max_thinking,False,___ (no response needed),True,familiarity_choice,fully_rational,no_choice_no_familiarity,1
1696,male,qwen_3_max_thinking,True,___,True,familiarity_choice,fully_rational,no_choice_no_familiarity,1
1697,male,qwen_3_max_thinking,True,"""ւ""",True,familiarity_choice,fully_rational,no_choice_no_familiarity,1
1698,male,qwen_3_max_thinking,False,___,True,familiarity_choice,fully_rational,no_choice_no_familiarity,1


## Функции для аналитики

In [39]:
def group_function(
    group_df: pd.DataFrame,
    condition_column: str,
    condition_control: str,
    metric_columns: list[str],
) -> pd.Series:
    if condition_column not in group_df.columns:
        raise RuntimeError(f'Column {condition_column} is not found in group DataFrame.')
    if group_df[group_df[condition_column] == condition_control].empty:
        raise RuntimeError(f'Condition {condition_control} is not found in {condition_column} column.')

    result = {}

    control_df = group_df[group_df[condition_column] == condition_control]
    control_stats = {}

    for metric in metric_columns:
        n = control_df[metric].count()
        avg = control_df[metric].mean()
        std = control_df[metric].std()

        control_stats[metric] = {'n': n, 'avg': avg, 'std': std}

        result[(metric, condition_control, 'N')] = int(n)
        result[(metric, condition_control, 'Avg')] = round(avg, 1) if pd.notna(avg) else np.nan
        result[(metric, condition_control, 'Std')] = round(std, 1) if pd.notna(std) else np.nan

    for condition in group_df[condition_column].dropna().unique():
        if condition == condition_control:
            continue

        condition_df = group_df[group_df[condition_column] == condition]

        for metric in metric_columns:
            n = condition_df[metric].count()
            avg = condition_df[metric].mean()
            std = condition_df[metric].std()

            result[(metric, condition, 'N')] = int(n)
            result[(metric, condition, 'Avg')] = round(avg, 1) if pd.notna(avg) else np.nan
            result[(metric, condition, 'Std')] = round(std, 1) if pd.notna(std) else np.nan

            control_avg = control_stats[metric]['avg']
            control_std = control_stats[metric]['std']
            control_n = control_stats[metric]['n']

            diff = avg - control_avg if pd.notna(avg) and pd.notna(control_avg) else np.nan
            r_diff = 100 * (avg / control_avg - 1) if pd.notna(control_avg) and control_avg != 0 else np.nan

            if (
                pd.notna(avg)
                and pd.notna(std)
                and pd.notna(control_avg)
                and pd.notna(control_std)
                and n > 1
                and control_n > 1
            ):
                t_stat, p_value = stats.ttest_ind_from_stats(
                    mean1=avg,
                    std1=std,
                    nobs1=n,
                    mean2=control_avg,
                    std2=control_std,
                    nobs2=control_n,
                    equal_var=False,
                )
            else:
                t_stat, p_value = np.nan, np.nan

            result[(metric, condition, 'Diff')] = round(diff, 1) if pd.notna(diff) else np.nan
            result[(metric, condition, 'R. Diff')] = round(r_diff, 1) if pd.notna(r_diff) else np.nan
            result[(metric, condition, 'T-Stat')] = round(t_stat, 2) if pd.notna(t_stat) else np.nan
            result[(metric, condition, 'P-Value')] = round(p_value, 4) if pd.notna(p_value) else np.nan

    return pd.Series(result)

In [40]:
def _sort_metric_condition_stat_columns(
    df: pd.DataFrame,
    condition_control: str,
) -> pd.DataFrame:
    if not isinstance(df.columns, pd.MultiIndex):
        return df

    stats_order_control = ['N', 'Avg', 'Std']
    stats_order_test = ['N', 'Avg', 'Std', 'Diff', 'R. Diff', 'T-Stat', 'P-Value']

    metrics = list(dict.fromkeys(df.columns.get_level_values(0)))
    conditions = list(dict.fromkeys(df.columns.get_level_values(1)))

    ordered_cols = []

    for metric in metrics:
        if condition_control in conditions:
            for stat in stats_order_control:
                col = (metric, condition_control, stat)
                if col in df.columns:
                    ordered_cols.append(col)

        for condition in conditions:
            if condition == condition_control:
                continue
            for stat in stats_order_test:
                col = (metric, condition, stat)
                if col in df.columns:
                    ordered_cols.append(col)

    remaining = [col for col in df.columns if col not in ordered_cols]
    ordered_cols.extend(remaining)

    return df.loc[:, ordered_cols]

In [99]:
def calc(
    df: pd.DataFrame,
    group_columns: list[str],
    condition_column: str,
    condition_control: str,
    metric_columns: list[str],
) -> pd.DataFrame:
    result = df.groupby(by=group_columns).apply(
        lambda g_df: group_function(g_df, condition_column, condition_control, metric_columns),
        include_groups=False
    )
    
    result.columns = pd.MultiIndex.from_tuples(result.columns)
    result = _sort_metric_condition_stat_columns(result, condition_control=condition_control)

    return result

In [42]:
def _pvalue_to_style(p_value: float, positive_effect: bool) -> str:
    if pd.isna(p_value):
        return ''

    if positive_effect:
        if p_value < 0.001:
            return 'background-color: #006d2c; color: white;'
        elif p_value < 0.01:
            return 'background-color: #31a354; color: white;'
        elif p_value < 0.05:
            return 'background-color: #74c476; color: black;'
        elif p_value < 0.10:
            return 'background-color: #c7e9c0; color: black;'
        else:
            return ''
    else:
        if p_value < 0.001:
            return 'background-color: #99000d; color: white;'
        elif p_value < 0.01:
            return 'background-color: #cb181d; color: white;'
        elif p_value < 0.05:
            return 'background-color: #fb6a4a; color: black;'
        elif p_value < 0.10:
            return 'background-color: #fcbba1; color: black;'
        else:
            return ''

In [43]:
def highlight_significance_gradient(
    df: pd.DataFrame,
    alpha: float = 0.10,
    stats_to_color: tuple[str, ...] = ('Diff', 'R. Diff', 'T-Stat', 'P-Value'),
    nonsig_style: str = 'background-color: #f2f2f2; color: #666666;',
) -> pd.DataFrame:
    '''
    Ожидает колонки вида: (metric, condition, stat)

    Логика:
    - p < alpha и Diff > 0  -> зеленый оттенок
    - p < alpha и Diff < 0  -> красный оттенок
    - p >= alpha            -> светло-серый
    '''
    styles = pd.DataFrame('', index=df.index, columns=df.columns)

    if not isinstance(df.columns, pd.MultiIndex):
        return styles

    metrics = df.columns.get_level_values(0).unique()
    conditions = df.columns.get_level_values(1).unique()

    for metric in metrics:
        for condition in conditions:
            p_col = (metric, condition, 'P-Value')
            diff_col = (metric, condition, 'Diff')

            if p_col not in df.columns or diff_col not in df.columns:
                continue

            for idx in df.index:
                pval = df.at[idx, p_col]
                diff = df.at[idx, diff_col]

                if pd.isna(pval) or pd.isna(diff):
                    continue

                if pval < alpha and diff != 0:
                    style = _pvalue_to_style(pval, positive_effect=(diff > 0))
                else:
                    style = nonsig_style

                for stat_name in stats_to_color:
                    col = (metric, condition, stat_name)
                    if col in df.columns:
                        styles.at[idx, col] = style

    return styles

In [44]:
def style_result(
    result_df: pd.DataFrame,
    alpha: float = 0.10,
):
    format_dict = {}

    for col in result_df.columns:
        stat = col[2]
        if stat == 'N':
            format_dict[col] = '{:.0f}'
        elif stat == 'Avg':
            format_dict[col] = '{:.1f}'
        elif stat == 'Std':
            format_dict[col] = '({:.1f})'
        elif stat == 'Diff':
            format_dict[col] = '{:.1f}'
        elif stat == 'R. Diff':
            format_dict[col] = '{:.1f}%'
        elif stat == 'T-Stat':
            format_dict[col] = '{:.2f}'
        elif stat == 'P-Value':
            format_dict[col] = '{:.4f}'

    table_styles = [
        {
            'selector': 'table',
            'props': [
                ('width', 'auto'),
                ('table-layout', 'auto'),
                ('border-collapse', 'collapse'),
            ],
        },
        {
            'selector': 'th',
            'props': [
                ('white-space', 'nowrap'),
                ('word-break', 'keep-all'),
                ('overflow-wrap', 'normal'),
                ('text-align', 'left'),
            ],
        },
        {
            'selector': 'td',
            'props': [
                ('white-space', 'nowrap'),
                ('word-break', 'keep-all'),
                ('overflow-wrap', 'normal'),
            ],
        },
    ]

    return (
        result_df.style
        .apply(highlight_significance_gradient, axis=None, alpha=alpha)
        .format(format_dict, na_rep='—')
        .set_table_styles(table_styles)
    )

## Результаты

In [101]:
style_result(
    calc(
        df=choice_df[choice_df['variation_id'] == 'default'],
        group_columns=['model_id', 'gender'],
        condition_column='condition_id',
        condition_control='no_choice',
        metric_columns=['would_buy_lottery_ticket']
    )
)

In [102]:
(choice_df['would_buy_lottery_ticket'] == True).isna().sum()

np.int64(0)

In [103]:
choice_df['condition_gender_id'] = choice_df['condition_id'] + '__' + choice_df['gender']

In [104]:
style_result(
    calc(
        df=choice_df[
            choice_df['would_buy_lottery_ticket']
             & (choice_df['variation_id'] != 'fully_rational')
        ],
        group_columns=['model_id', 'variation_id'],
        condition_column='condition_gender_id',
        condition_control='no_choice__male',
        metric_columns=['requested_lottery_ticket_price']
    )
)

In [106]:
choice_df[
    choice_df['would_buy_lottery_ticket'] & (choice_df['model_id'] == 'deepseek_v3.2')
].groupby(by=['model_id', 'variation_id', 'condition_gender_id']).count()

gender  \
model_id      variation_id   condition_gender_id           
deepseek_v3.2 default        choice__female           18   
                             choice__male             34   
                             no_choice__female        19   
                             no_choice__male          39   
              fully_rational choice__female            2   
                             choice__male              3   
                             no_choice__female         3   
                             no_choice__male           4   
              large_pot      choice__female           17   
                             choice__male             41   
                             no_choice__female        19   
                             no_choice__male          32   
              letter_ticket  choice__female           22   
                             choice__male             41   
                             no_choice__female        27   
                             no_choice__male          36   
              rephrasing     choice__female           19   
                             choice__male             34   
                             no_choice__female        16   
                             no_choice__male          29   

                                                  would_buy_lottery_ticket  \
model_id      variation_id   condition_gender_id                             
deepseek_v3.2 default        choice__female                             18   
                             choice__male                               34   
                             no_choice__female                          19   
                             no_choice__male                            39   
              fully_rational choice__female                              2   
                             choice__male                                3   
                             no_choice__female                           3   
                             no_choice__male                             4   
              large_pot      choice__female                             17   
                             choice__male                               41   
                             no_choice__female                          19   
                             no_choice__male                            32   
              letter_ticket  choice__female                             22   
                             choice__male                               41   
                             no_choice__female                          27   
                             no_choice__male                            36   
              rephrasing     choice__female                             19   
                             choice__male                               34   
                             no_choice__female                          16   
                             no_choice__male                            29   

                                                  chosen_ticket  \
model_id      variation_id   condition_gender_id                  
deepseek_v3.2 default        choice__female                  18   
                             choice__male                    34   
                             no_choice__female                0   
                             no_choice__male                  0   
              fully_rational choice__female                   2   
                             choice__male                     3   
                             no_choice__female                0   
                             no_choice__male                  0   
              large_pot      choice__female                  17   
                             choice__male                    41   
                             no_choice__female                0   
                             no_choice__male                  0   
              letter_ticket  choice__female                  22   
       